# PakNotes DB in Colab

This notebook lets you work with the `english_mcqs.db` SQLite database from Google Colab.

It supports two workflows:
1. Mount Google Drive and point to a database already stored there.
2. Upload the database file directly from your computer into the Colab session.


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd


## Option 1: Mount Google Drive

Run this cell if your database file is already in Google Drive.

Example path after mounting:
`/content/drive/MyDrive/PakNotesDB/data/english_mcqs.db`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Set `db_path` below if you mounted Google Drive. Skip this cell if you plan to upload the DB instead.


In [ ]:
db_path = Path('/content/drive/MyDrive/PakNotesDB/data/english_mcqs.db')
print(db_path)
print('Exists:', db_path.exists())


## Option 2: Upload the Database File

Run this cell instead if the DB is on your computer and not in Google Drive.


In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_name = next(iter(uploaded))
db_path = Path('/content') / uploaded_name
print(db_path)
print('Exists:', db_path.exists())


## Connect to SQLite

This cell requires `db_path` to already be set by either the Drive flow or the upload flow.


In [ ]:
if 'db_path' not in globals():
    raise ValueError('Set db_path first by using either the Drive or upload section above.')

conn = sqlite3.connect(db_path)
print('Connected to:', db_path)


## List Available Tables


In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
tables


## Preview MCQs


In [ ]:
mcqs_preview = pd.read_sql_query(
    'SELECT * FROM mcqs LIMIT 10',
    conn,
)
mcqs_preview


## Row Counts for Key Tables


In [ ]:
count_query = """
SELECT 'subjects' AS table_name, COUNT(*) AS row_count FROM subjects
UNION ALL
SELECT 'subcategories', COUNT(*) FROM subcategories
UNION ALL
SELECT 'mcqs', COUNT(*) FROM mcqs
UNION ALL
SELECT 'mcqs_clean', COUNT(*) FROM mcqs_clean
UNION ALL
SELECT 'imports', COUNT(*) FROM imports
UNION ALL
SELECT 'pipeline_logs', COUNT(*) FROM pipeline_logs
UNION ALL
SELECT 'page_fetch_log', COUNT(*) FROM page_fetch_log
"""

pd.read_sql_query(count_query, conn)


## Example Custom Query

Edit the SQL below to explore the database however you want.


In [ ]:
query = '''
SELECT *
FROM mcqs_clean
LIMIT 20
'''

pd.read_sql_query(query, conn)


## Close the Connection


In [ ]:
conn.close()
print('Connection closed.')
